# 第 7 章 · 动态工作流基础：用 Python 编排 Agent

> 复用第四章“大纲 → 初稿 → 润色”的场景，学习 `Workflow + @node + ctx.run_node()`。本章集中解决：顺序、传值、条件、迭代和有界并发。人工确认、持久恢复、execution ID 和副作用幂等移到下一章。

使用仓库锁定的 **google-adk 2.6.3 / Python 3.12**。从项目根目录运行 `uv sync`，再运行 `uv run --env-file .env jupyter lab`；缺少 `DEEPSEEK_API_KEY` 时直接报错，不跳过真实模型验证。按顺序执行全部单元格，不需要其他 Notebook 的内存变量。

本章真实调用 DeepSeek：三步写作流水线加一次受控修改，共 **4 次模型调用步骤**。SDK 网络重试另计。运行输出和章末验收均保存到 Notebook 中。

---

## 1. 从第四章迁移：变的是编排接口

| 第四章 | 本章 | 变化 |
|---|---|---|
| `SequentialAgent(sub_agents=[A, B, C])` | 依次 `await ctx.run_node(A/B/C, input)` | 顺序由 Python 表达 |
| `output_key` + instruction 的 `{key}` | 返回值 + 显式输入 | 局部数据不必全部经过 State |
| `ParallelAgent` | parallel worker / `asyncio.gather` | 批次数量可由输入决定 |
| `LoopAgent` + `escalate` | 有上限的 Python 循环 | 退出条件和失败结果直接可见 |

一句话概括这张表：**第四章把“走哪几步”写成了一份数据（节点列表），本章把它写成了一段程序（Python 函数体）。**

### 1.1 「动态」到底指什么？

“动态”这个词在 Agent 领域至少有三层意思，非常容易混：

| 「动态」可能指 | 谁在决定路径 | 在 ADK 里对应什么 | 本章的 `Workflow` 是它吗 |
|---|---|---|---|
| ① **模型自主规划** | LLM 临场推理决定下一步 | 第四章的 `transfer_to_agent`、`sub_agents` | **不是** |
| ② **运行时创建 Agent** | 系统边跑边生成新的 Agent | 一些多智能体框架的“动态组队” | **不是**（也不要求） |
| ③ **用代码表达路径** | 你写的 Python | `Workflow` + `@node` + `ctx.run_node()` | ✅ **就是它** |

所以本章说的“动态”，**不是“AI 自己做主”，而是“控制权在你手里”**——路径由你的 Python 代码走出来，不由模型临场决定。

ADK 源码里的说法更精确。`Workflow` 的 `max_concurrency` 字段上写着：

> **Dynamic nodes (via `ctx.run_node()`) are excluded** —— 它们由父节点内联等待。

也就是说，「动态节点」的官方定义是：**由父节点在运行时通过 `ctx.run_node()` 调起来的节点**。与它相对的是 **graph-scheduled nodes**——由 `edges` 声明、构造时编译成图并被校验的那些节点。

> 📌 **所以「动态」的对立面不是「图」，而是「预先声明的边」。** 本章的 `Workflow` 仍然是图工作流（ADK 官方文档也把 *Dynamic workflows* 放在 *Graph Workflows* 大类下面），只是**边不由你声明，而由代码走出来**。

这件事在代码里看得很清楚。本章的主流水线只声明了 **一条边**：

```python
pipeline = Workflow(name="pipeline", edges=[("START", pipeline_entry)])
```

声明出来的图里只有“入口”这**一个节点**；真正的三步写作、条件分支、有界循环，全在 `pipeline_entry` / `writing_pipeline` 的函数体里。换句话说：**如果打开 `adk web` 的 Graph 面板看这个工作流，它画不出什么——因为流程根本不在图里，在代码里。**（第 8 章那个跨三进程的综合工程同样只声明了一条边。）

### 1.2 两个正交的维度：谁表达路径 × 谁作决策

不要把“动态”和“模型自主”混为一谈，它们是**两个独立的维度**：

- **图还是代码** → 决定**路径如何表达**（本章选“代码”）；
- **规则还是 LLM** → 决定**谁作决策**（本章的检查用规则，写作交给 LLM）。

两者可以自由组合。本章落在 **“代码 + 规则”** 这一格：用 `review["passed"]` 这样的规则字段在判断。但你完全可以写成 **“代码 + LLM”** ——让 `ctx.run_node()` 调起的那个 Agent 在自己内部跑 ReAct 循环。

> 📌 **区别在层级**：Agent 内部的 ReAct 只决定“**这一步内部怎么做**”，不决定“**下一步是谁**”。下一步始终由你 `return` 什么、`if` 判什么来决定。要让模型决定“下一步是谁”，那是第四章的 `transfer_to_agent`。

Graph workflows 也支持分支、循环和并行；LLM 委派属于另一种协作方式。另外，新 `Workflow` 在 2.6.3 中不能直接充当 `LlmAgent.sub_agents`，本章把它作为 Runner 的根对象——原因藏在类型继承里：`BaseAgent` 继承 `BaseNode`，而 `Workflow` 只继承 `BaseNode`。**它是“能被调度的一步”，但不是“有自主性的智能体”。**

### 1.3 这个选择的三个后果

把路径从“数据”变成“程序”，会带来三个你需要提前知道的后果。它们都不是 Bug，而是这个设计的必然：

1. **跑之前画不出流程图。** 本章后面的 `node_info.path` 是执行时拼出来的（类似 `hello@1/hello_entry@1`），不是谁声明的。第 2 节会看到它的具体形态。
2. **并发上限只能挂在节点上。** 工作流无从知道同时会有几个节点在跑，所以框架的 `max_concurrency` 明确**不适用于**动态节点——它们是父节点内联 `await` 的，再对它们限流会造成死锁。第 5 节会说明为什么并发实验写的是 `max_parallel_workers`。
3. **恢复不能“从函数中间继续”。** Python 没法从函数体某一行接着执行，所以第 8 章的恢复要靠“父函数重入 + 子节点回放”。

还有一条代价值得记住：静态图能在**构造时**就报出环、不可达节点、相邻节点 schema 不匹配这类错误；**动态工作流只能等跑起来才知道**。所以本章和第 8 章都要自己立契约、自己写断言——框架帮不了你。

```mermaid
flowchart LR
    I[主题] --> O[大纲 Agent]
    O --> W[写作 Agent]
    W --> P[润色 Agent]
    P --> C[Python 检查]
    C -->|通过| F[交付]
    C -->|未通过且有预算| R[修改 Agent]
    R --> C
```

> 📌 上面这张图**不是** `edges` 声明出来的图，而是**函数体走出来的路径**——它是画给人看的，不是给框架看的。框架看到的声明图只有 `START → pipeline_entry` 这一条边。

参考：[官方 Dynamic workflows](https://adk.dev/graphs/dynamic/)。

## 2. 最小运行骨架：先认识三个构件

- `@node`：把函数登记为框架节点。
- `ctx.run_node()`：执行子节点并取得返回值。
- `Workflow(edges=[("START", entry)])`：把 Runner 入口连接到编排函数。

调用子节点的父函数设置 `rerun_on_resume=True`；普通叶子节点保持默认值。默认参数绑定中，`ctx` 和 `node_input` 是特殊接口名，其他参数名会从 State 取值。

> 📌 `@node` 不只是"贴个装饰器"。它是你和框架之间的**登记动作**：只有登记过的函数，框架才认识它、才愿意在 checkpoint 里为它保存结果。把一个普通函数直接 `await` 掉，框架看不见这一层，后面的"回放"也就无从谈起。

### 2.1 `rerun_on_resume` 到底是什么？

上面那句"父函数设 `True`、叶子保持默认"是一条规则。规则要能用，得先知道它背后的模型。现在只需要理解到**够用**的程度，完整机制在第 8 章展开：

- 一个节点如果**内部调用了别的节点**（也就是里面有 `ctx.run_node(...)`），它就是"父函数"。
- 当流程因为人工确认、故障或重启而**恢复**时，框架**不是**从断点"续上"，而是**重新进入父函数、从头再跑一遍**。
- 但已经成功完成的子节点**不会重新执行**——框架从 checkpoint（检查点）里把上次的结果直接还给父函数。
- 所以父函数里那些"取输入、算路径、拼参数"的普通代码**必须能重复执行**；真正会改变外部世界的动作（写库、发请求、存文件）要放进**叶子节点**。

> ⚠️ 这里先记住结论就好。第 8 章会给你**证据**：那里用执行计数证明父函数确实重入了、而子节点确实只执行了一次。

In [1]:
import asyncio
import importlib.metadata as metadata
import json
import os
from uuid import uuid4

# 使用随 LiteLLM 安装的成本表，不影响真实模型请求。
os.environ["LITELLM_LOCAL_MODEL_COST_MAP"] = "True"

from google.adk import Context, Workflow
from google.adk.agents import Agent
from google.adk.models.lite_llm import LiteLlm
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.adk.workflow import node
from google.genai import types
from pydantic import BaseModel, Field

assert os.getenv("DEEPSEEK_API_KEY"), "请先配置 DEEPSEEK_API_KEY，再执行本章。"
assert metadata.version("google-adk") == "2.6.3", (
    "请从项目根目录执行 uv sync。"
)
APP, USER = "adk_ch07", "student"
checks = {}
print("运行环境：", metadata.version("google-adk"), "；DeepSeek 密钥已配置")

运行环境： 2.6.3 ；DeepSeek 密钥已配置


Runner 接收 `types.Content`。下面只负责转换入口和收集返回值，先不加入日志、重试或恢复协议。

**输出为什么不能只用 `is_final_response()`？** 子节点也会生成最终事件。`node_info.output_for` 列出输出的归属路径；没有 `/` 的路径指向根 Workflow，例如 `hello@1`。本节选取根输出，第 3 节再观察完整路径。

> 📌 这里定义的 `input_text` / `collect` / `start` 三个辅助函数，本章后面所有实验都会复用。现在不必逐行细读，只需要知道它们各自负责一件事：**归一入口**（`input_text`）、**按归属取根输出**（`collect`）、**把“建会话 → 跑一次”打包成一个 `await`**（`start`）。

> 📌 **顺带看清 `hello@1` 这个写法**：`hello` 是工作流名，`@1` 是本次执行里的调用序号——**同一个节点被调第二次就是 `@2`**。完整路径会更长，第 3 节你会看到类似 `hello@1/hello_entry@1/normalize_request@1` 的形态。

> 为什么路径要带序号？因为这正是 1.1 节说的“路径是执行出来的”：框架没法提前知道某个节点会被调用几次，只能边跑边记。

In [2]:
def input_text(value) -> str:
    if isinstance(value, types.Content):
        return "".join(part.text or "" for part in value.parts or [])
    if isinstance(value, str):
        return value
    raise TypeError(f"入口需要 Content 或 str，实际是 {type(value).__name__}")


async def collect(runner, session_id, message, *, invocation_id=None):
    output, events = None, []
    async for event in runner.run_async(
        user_id=USER,
        session_id=session_id,
        new_message=message,
        invocation_id=invocation_id,
    ):
        events.append(event)
        if event.node_info and event.output is not None:
            if any(
                "/" not in path for path in event.node_info.output_for or []
            ):
                output = event.output
    return output, events


async def start(
    workflow, payload, *, session_service=None, artifact_service=None
):
    service = session_service or InMemorySessionService()
    session_id = uuid4().hex
    await service.create_session(
        app_name=APP, user_id=USER, session_id=session_id
    )
    runner = Runner(
        agent=workflow,
        app_name=APP,
        session_service=service,
        artifact_service=artifact_service,
    )
    text = (
        payload
        if isinstance(payload, str)
        else json.dumps(payload, ensure_ascii=False)
    )
    message = types.Content(role="user", parts=[types.Part(text=text)])
    output, events = await collect(runner, session_id, message)
    return output, events, runner, session_id

In [3]:
@node
def normalize_topic(node_input: str) -> str:
    return node_input.strip().removeprefix("主题：")


@node(rerun_on_resume=True)
async def hello_entry(ctx: Context, node_input):
    topic = await ctx.run_node(normalize_topic, input_text(node_input))
    return f"学习主题：{topic}"


hello = Workflow(name="hello", edges=[("START", hello_entry)])
result, events, *_ = await start(hello, "  主题：动态工作流  ")
assert result == "学习主题：动态工作流"
checks["最小工作流与根输出"] = True
for event in events:
    if event.node_info and event.output is not None:
        print(
            event.node_info.path,
            "→",
            event.node_info.output_for,
            "→",
            event.output,
        )

hello@1/hello_entry@1/normalize_topic@1 → ['hello@1/hello_entry@1/normalize_topic@1'] → 动态工作流
hello@1/hello_entry@1 → ['hello@1/hello_entry@1', 'hello@1'] → 学习主题：动态工作流


## 3. 真实 DeepSeek 流水线：大纲 → 初稿 → 润色

使用第四章的同一主题“程序员为什么需要学习写作”。这次不写 `output_key`，而是将上一步结果显式传给下一步。

沿用第二章的 DeepSeek 兼容方案：`response_format={"type": "json_object"}` 加本地 Pydantic 校验。JSON Output 保证的是输出格式；字段由 schema 校验，业务要求下一节另行检查。不设置 `output_schema`，避免请求被转换为不兼容的 `json_schema`。

模型回调只记录 Agent 名和 token 数，不修改结果；执行路径从 Event 读取。后续可以用它证明调用确实发生，以及恢复后没有重复请求模型。

> 📌 这一段是本章第一次真实花钱调用模型。建议边读边对照三件事：**Python 决定顺序**（三次 `ctx.run_node` 依次写下来）、**模型只负责单步**（每个 Agent 只做"列要点 / 写正文 / 润色"中的一件）、**数据显式流动**（上一步的返回值当下一步的入参，不经过 State）。

> ⚠️ 注意这里**不设置** `output_schema`：它是为了避免请求被转换成 DeepSeek 不兼容的 `json_schema`。所以 JSON 的正确性由本地 Pydantic 校验兜底——**格式靠 `response_format` 保证，字段靠校验保证，业务要求还要另外检查**（第 4 节）。

In [4]:
model_responses = []


def record_response(callback_context, llm_response):
    usage = llm_response.usage_metadata
    model_responses.append(
        {
            "agent": callback_context.agent_name,
            "tokens": usage.total_token_count if usage else None,
        }
    )
    return None


def deepseek():
    return LiteLlm(
        model="deepseek/deepseek-chat",
        max_tokens=768,
        response_format={"type": "json_object"},
    )


class Outline(BaseModel):
    points: list[str] = Field(min_length=3, max_length=3)


class Draft(BaseModel):
    title: str = Field(min_length=1)
    body: str = Field(min_length=1)


outliner = Agent(
    name="outliner",
    model=deepseek(),
    mode="single_turn",
    instruction="根据输入 topic 和 notes 列出三个写作要点，遵循 notes 的事实。"
    '只输出 JSON：{"points": ["要点1", "要点2", "要点3"]}。',
    after_model_callback=record_response,
)
writer = Agent(
    name="writer",
    model=deepseek(),
    mode="single_turn",
    instruction="根据 topic、notes 和 outline 写中文入门短文。正文目标 80～100 个字符，"
    "验收范围 60～120 个字符，标点和英文字母也计数。只解释核心机制，"
    "使用 required_terms 中的每个词，包含以‘例如’引出的实际例子。"
    "例子使用内容写作或审批场景，不引入支付等额外业务。"
    '遵循 notes，只输出 JSON：{"title": "标题", "body": "正文"}。',
    after_model_callback=record_response,
)
polisher = Agent(
    name="polisher",
    model=deepseek(),
    mode="single_turn",
    instruction="你是保守的中文编辑。润色输入 draft；若有 findings，只修复所列问题，"
    "已经满足的规则不要破坏。长度已合格时保持原长度，不要压缩。遵循 request.notes，"
    "正文目标 80～100 个字符，验收范围 60～120，标点和英文字母也计数。"
    "正文写成四句完整句子，每句约 20～25 字符：核心观点、作用说明、"
    "例如场景、实际效果。过短的稿件必须扩写，不能只替换字词或压成两句。"
    "包含 request.required_terms 和‘例如’实际例子。"
    "例子限于内容写作或审批，不引入支付等额外业务。"
    '保留准确含义，只输出 JSON：{"title": "标题", "body": "正文"}。',
    after_model_callback=record_response,
)

> 📌 注意 `mode="single_turn"` 这一行。**显式写出来是“声明意图”，并不是必需**：当一个 Agent 作为 Workflow 节点被 `ctx.run_node()` 调用时，ADK 的默认模式本来就是 `single_turn`。

ADK 里 `mode` 有三种取值：

| 取值 | 含义 | 典型场景 |
|---|---|---|
| `chat` | 可被 `transfer_to_agent` 转交的对话 Agent | 子 Agent、需要互相转交 |
| `task` | 与用户多轮对话来完成任务 | 需要追问澄清的任务型 Agent |
| `single_turn` | 完成一次任务，不与用户对话 | **Workflow 里的节点** |

> 📌 **`chat` 和 `single_turn` 的真正差别不是“参不参与对话”，而是输入从哪来**：
>
> | 模式 | 拿什么当输入 |
> |---|---|
> | `chat` | **会话历史**（session events） |
> | `single_turn` | **上游节点直接传来的 `node_input`** |
>
> 这是两种不同的**数据来源**。所以当一个 `chat` 模式的 Agent 被接在别的节点后面时，ADK 在**构造工作流图的时候就会直接拒绝**：
>
> > The agent '...' has been added to the workflow with mode='chat' following node '...'. This is not supported because **chat-mode agents rely on conversational history (session events) and cannot consume direct node inputs from preceding nodes.** Please change the agent's mode to 'single_turn'

本章的 Agent 都是被 `ctx.run_node()` 调起来的（走 1.1 节说的动态调度），默认落到的就是 `single_turn`，所以这里显式写出**不是必需，但很值**——它把“这个 Agent 吃的是节点输入，不是聊天记录”写在了脸上。参数全貌见第 2 章。

In [5]:
@node(rerun_on_resume=True)
async def writing_pipeline(ctx: Context, node_input: dict) -> dict:
    outline = Outline.model_validate_json(
        await ctx.run_node(outliner, node_input)
    )
    draft = Draft.model_validate_json(
        await ctx.run_node(
            writer,
            {**node_input, "outline": outline.model_dump()},
        )
    )
    final = Draft.model_validate_json(
        await ctx.run_node(
            polisher,
            {"request": node_input, "draft": draft.model_dump()},
        )
    )
    return {"outline": outline.model_dump(), "draft": final.model_dump()}


@node(rerun_on_resume=True)
async def pipeline_entry(ctx: Context, node_input):
    return await ctx.run_node(
        writing_pipeline, json.loads(input_text(node_input))
    )


pipeline = Workflow(name="pipeline", edges=[("START", pipeline_entry)])
writing_request = {
    "topic": "程序员为什么需要学习写作",
    "notes": "写作有助于澄清需求、记录设计理由、减少团队沟通成本。",
    "required_terms": ["例如"],
}
before = len(model_responses)
pipeline_result, pipeline_events, *_ = await start(pipeline, writing_request)
assert len(model_responses) - before == 3
assert [row["agent"] for row in model_responses[before:]] == [
    "outliner",
    "writer",
    "polisher",
]
agent_paths = [
    event.node_info.path
    for event in pipeline_events
    if event.node_info
    and event.content
    and event.author in {"outliner", "writer", "polisher"}
]
checks["真实模型流水线与执行顺序"] = True
print(json.dumps(pipeline_result, ensure_ascii=False, indent=2))
print("实际模型调用路径：", *agent_paths, sep="\n")

{
  "outline": {
    "points": [
      "写作有助于澄清需求：程序员在写需求文档或注释时，必须用自然语言明确表达功能边界与预期，这一过程本身就能暴露理解上的模糊或矛盾之处，从而提前消除歧义。",
      "写作有助于记录设计理由：通过撰写设计文档或决策记录（例如记录为什么选择某个技术方案而放弃其他方案），团队可以保留技术决策的上下文，避免未来重复讨论或误解。",
      "写作有助于减少团队沟通成本：清晰、结构化的书面材料（例如技术方案说明、接口文档、会议纪要）可以让成员异步获取信息，减少反复口头解释和会议时间，提升协作效率。"
    ]
  },
  "draft": {
    "title": "程序员为何要学写作",
    "body": "写作能帮程序员澄清需求并理清思路。它促使设计理由被记录与共享。例如写接口文档时常发现遗漏。这样能显著减少团队沟通成本。"
  }
}
实际模型调用路径：
pipeline@1/pipeline_entry@1/writing_pipeline@1/outliner@1
pipeline@1/pipeline_entry@1/writing_pipeline@1/writer@1
pipeline@1/pipeline_entry@1/writing_pipeline@1/polisher@1


> 📌 `ctx.run_node()` 是**调用—返回**，`transfer_to_agent` 是**移交—接管**。前者执行完会把控制权还给你，后者交出去就不回来了。本例的三次模型调用只负责完成写作任务，先做哪一步、后做哪一步，完全由上面的 Python 代码决定。

> ⚠️ **返回类型不要混为一谈**：函数节点返回 Pydantic 模型时会变成**字典**；Agent 节点在没设置 `output_schema` 时返回的是 **JSON 字符串**。把两者当同一种东西处理，是本章最常见的报错来源。

State 仍适合工具/回调共享的摘要、跨步骤流程状态和 instruction 插值。普通中间结果优先显式传值；局部变量的恢复原理在第八章解释。

---

## 4. 条件与有界迭代：检查不通过，就修改

继续同一写作场景。业务规则检查标题、正文长度、必要术语和实际例子的标记。

> 📌 **检查不通过不是"程序出错"。** `passed=False` 是一个**业务结论**，它应该走进修改分支，而不是触发网络重试或抛异常。把"业务判断"和"运行故障"分开，是让流程行为可预期的关键。

`max_revisions` 表示最多修改几次，初次检查不占修改预算。达到上限必须返回 `revision_limit`，不能把最后一稿自动标成成功。

In [6]:
class WritingRequest(BaseModel):
    topic: str = Field(min_length=1)
    notes: str = Field(min_length=1)
    required_terms: list[str] = Field(default_factory=list)


@node
def review_article(node_input: dict) -> dict:
    request = WritingRequest.model_validate(node_input["request"])
    draft = Draft.model_validate(node_input["draft"])
    findings = []
    if not draft.title.strip():
        findings.append("标题不能为空白")
    if not 60 <= len(draft.body) <= 120:
        findings.append(f"正文 {len(draft.body)} 字，要求 60～120 字")
    for term in dict.fromkeys(["例如", *request.required_terms]):
        if term not in draft.body:
            findings.append(f"正文必须包含：{term}")
    return {"passed": not findings, "findings": findings}


@node(rerun_on_resume=True)
async def refine_article(ctx: Context, node_input: dict) -> dict:
    request, draft = node_input["request"], node_input["draft"]
    budget = node_input["max_revisions"]
    if not isinstance(budget, int) or isinstance(budget, bool) or budget < 0:
        raise ValueError("max_revisions 必须是非负整数")
    for revision in range(budget + 1):
        review = await ctx.run_node(
            review_article, {"request": request, "draft": draft}
        )
        if review["passed"] or revision == budget:
            return {
                "draft": draft,
                **review,
                "revisions": revision,
                "stop_reason": "passed"
                if review["passed"]
                else "revision_limit",
            }
        draft = Draft.model_validate_json(
            await ctx.run_node(
                polisher,
                {
                    "request": request,
                    "draft": draft,
                    "findings": review["findings"],
                },
            )
        ).model_dump()


@node(rerun_on_resume=True)
async def refinement_entry(ctx: Context, node_input):
    return await ctx.run_node(
        refine_article, json.loads(input_text(node_input))
    )


refinement = Workflow(name="refinement", edges=[("START", refinement_entry)])

In [7]:
valid_draft = {
    "title": "程序员的写作练习",
    "body": "写作可以帮助程序员梳理需求、说明设计理由并减少沟通误差。例如，在接口变更前写清输入、输出与兼容范围，团队就能及时发现遗漏，也方便后来维护的人理解当时的选择。",
}
missing_example = {
    **valid_draft,
    "body": valid_draft["body"].replace("例如", "比如"),
}
cases = [
    (valid_draft, 0, True),
    ({"title": "初稿", "body": "太短"}, 0, False),
    (missing_example, 1, True),
]
for draft, budget, expected in cases:
    result, *_ = await start(
        refinement,
        {
            "request": writing_request,
            "draft": draft,
            "max_revisions": budget,
        },
    )
    print(json.dumps(result, ensure_ascii=False))
    assert result["passed"] is expected, result
    assert result["revisions"] <= budget
checks["一次通过、预算耗尽与真实模型修改"] = True

{"draft": {"title": "程序员的写作练习", "body": "写作可以帮助程序员梳理需求、说明设计理由并减少沟通误差。例如，在接口变更前写清输入、输出与兼容范围，团队就能及时发现遗漏，也方便后来维护的人理解当时的选择。"}, "passed": true, "findings": [], "revisions": 0, "stop_reason": "passed"}
{"draft": {"title": "初稿", "body": "太短"}, "passed": false, "findings": ["正文 2 字，要求 60～120 字", "正文必须包含：例如"], "revisions": 0, "stop_reason": "revision_limit"}


{"draft": {"title": "程序员的写作练习", "body": "程序员需要学习写作来澄清需求和记录设计理由。写作能说明设计取舍并有效减少团队沟通成本。例如在接口变更前写清输入输出与兼容范围。这能暴露遗漏并让后续维护者理解选择。"}, "passed": true, "findings": [], "revisions": 1, "stop_reason": "passed"}


第三次运行将已合格正文中的“例如”改为“比如”，制造一个明确缺陷，真实调用 DeepSeek 修复，再重新检查；前两次不调用模型。这样能够分别观察**编排规则**和**模型行为**。修改只应修复问题，不能破坏已经合格的长度。

### 4.1 为什么提示词里的目标要比验收范围更严？

提示词里写的是 **80～100 字符**，而代码验收用的是 **60～120 字符**——这两个数字故意不一样：

- 模型的输出长度**天生有抖动**。如果把验收线直接写进提示词（"必须 60～120 字"），模型会贴着边界输出，一半样本会掉到范围外，本章验收就会**随机失败**。
- 所以提示词里给一个**更窄的目标区间**，验收用**更宽的硬边界**，中间留出抖动余量。

> 💡 **如果模型仍然不达标怎么办？** 本章的验收会**如实失败**：`stop_reason` 返回 `revision_limit`，最后一稿不会被标成成功。这不是 Bug，是设计。处理方式按推荐顺序：① 收紧提示词里的目标区间；② 提高 `max_revisions`；③ 放宽规则检查到**业务真正在意**的口径。**不要**为了"让 Notebook 变绿"就把检查改宽到总是通过——那等于把验收作废。

标点和英文字母均按 Python `len()` 计数。

> ⚠️ **检查边界**：这些规则只证明标题、长度和术语符合要求，**不是事实评估**。尤其不能把"包含 Python"当作"准确理解 Dynamic workflows"。第 6 节会给模型提供已核对的事实，并展示全文供人工审核。

---

## 5. 动态批处理：验证并发上限，而不只看耗时

任务数由输入列表决定。`node(worker, parallel_worker=True, max_parallel_workers=2)` 最多同时运行两个工作节点，结果按输入顺序返回。

> 📌 这里绕了一下用 `node(...)` 再包一层，是为了把 `parallel_worker` 这类**节点级配置**挂上去。
>
> **为什么并发上限挂在节点上，而不是挂在工作流上？** 因为 1.3 节说的第 2 条：动态节点是父节点内联 `await` 的，工作流本来就不知道同时有几个在跑，所以框架的 `max_concurrency` 明确把动态节点排除在外——对已经在等的父节点限流会造成死锁。**并发控制只能落在「被调的那个节点」身上。**
>
> 下面在真实 ADK 节点内记录「当前有几个在跑」，用来证明两件事：并发**确实发生**了，而且**没有超过上限**——只看总耗时是证明不了这个的。

In [8]:
concurrency = {"active": 0, "peak": 0}


@node
async def inspect_topic(node_input: str) -> dict:
    concurrency["active"] += 1
    concurrency["peak"] = max(concurrency["peak"], concurrency["active"])
    try:
        await asyncio.sleep(0.03)
        return {"topic": node_input, "characters": len(node_input)}
    finally:
        concurrency["active"] -= 1


bounded_inspector = node(
    inspect_topic, parallel_worker=True, max_parallel_workers=2
)


@node(rerun_on_resume=True)
async def inspect_batch(ctx: Context, node_input):
    return await ctx.run_node(
        bounded_inspector, json.loads(input_text(node_input))
    )


topics = ["节点", "循环", "checkpoint", "人工确认", "状态"]
inspector = Workflow(name="inspector", edges=[("START", inspect_batch)])
concurrency.update(active=0, peak=0)
result, *_ = await start(inspector, topics)
assert concurrency == {"active": 0, "peak": 2}
assert [item["topic"] for item in result] == topics
checks["并发确实发生且不超过上限"] = True
print(
    "实测并发峰值：",
    concurrency["peak"],
    "；返回顺序：",
    [r["topic"] for r in result],
)

实测并发峰值： 2 ；返回顺序： ['节点', '循环', 'checkpoint', '人工确认', '状态']


不同节点并行时也可以用 `await asyncio.gather(ctx.run_node(A, ...), ctx.run_node(B, ...))`。

> ⚠️ **两个常见误解**：① **逐项 `await` 就是串行**——写了 `async` 不等于自动并行；② **CPU 密集任务不会因为加了 `async` 就自动并行**——`asyncio` 是单线程协作式调度，它让"等 I/O"的时间重叠，不让"算"的时间重叠。

本章选择“执行异常使整批失败；业务不合格返回检查结果”的策略。并发分支优先各自返回结果，由父节点汇总；写同一个 State key 不会自动合并。不要用 `except BaseException` 或 `gather(return_exceptions=True)` 吞掉人工暂停信号。

> 📌 `max_parallel_workers` 只约束**本批次**的节点并发，**不等于你的 API 配额**。批次嵌套时上限会叠乘，生产环境要另外做全局限流。

## 6. 与 LangGraph 对照 🔄

| 本章构件 | LangGraph 对照 | 差异 |
|---|---|---|
| `Workflow(edges=...)` | Graph API 的 nodes / edges | 路由与状态协议不同 |
| 编排函数 + `ctx.run_node()` | Functional API 的 `entrypoint` / `task` | 都用代码表达控制流；装饰器与恢复协议不同 |
| 返回值与 `ctx.state` | 任务返回值与 State | **并发写入不自动套用 reducer** |
| `parallel_worker` | Map / 并行任务 | 上限只约束本批次，服务级配额另行控制 |

**对照阅读**：建议翻到 [04-LangGraph核心-图与状态](../langchain-langgraph/04-LangGraph核心-图与状态.ipynb) 的 §3「Reducer」和 §4「条件边」对照着看——那里解释了"并发写入为什么需要 reducer"，正好反过来解释本章"为什么显式传值比写 State 更省心"。

参考：[Dynamic workflows](https://adk.dev/graphs/dynamic/) · [Data handling](https://adk.dev/graphs/data-handling/) · [DeepSeek JSON Output](https://api-docs.deepseek.com/guides/json_mode/) · [LangGraph Functional API](https://docs.langchain.com/oss/python/langgraph/functional-api)。

---

## 📌 本章要点回顾

- **"动态"= 用 Python 表达路径**：它的对立面是「预先声明的边」而不是「图」（§1.1）；条件、循环、函数调用决定下一步；不要求模型自主规划，也不要求运行时创建新 Agent；
- **三个构件**：`@node` 登记函数、`ctx.run_node()` 调用并取回值、`Workflow(edges=[("START", entry)])` 把 Runner 入口接上；
- **显式传值优于 State**：普通中间结果直接当参数传；State 留给工具/回调共享、跨步骤流程状态和 instruction 插值；
- **业务不合格 ≠ 运行异常**：`passed=False` 走修改分支，预算耗尽必须如实返回 `revision_limit`；
- **并发要用数据证明**：记录峰值验证"确实并发且不超上限"，只看耗时证明不了；
- **`rerun_on_resume` 的完整机制在第 8 章**，本章先记住"父函数要能重跑，副作用放叶子节点"。

---

## 🧪 本章练习

### 1. 给最小骨架加一个参数（基础）

让 `hello` 工作流接收 `{"topic": "...", "audience": "..."}`，在 `hello_entry` 里解包后传给 `normalize_topic`，最后返回一个包含两个字段的字典。

**验收**：断言根输出里**同时**含有主题和读者；打印一次运行中**所有**事件的 `node_info.path` 与 `node_info.output_for`。

### 2. 读懂 `output_for` 与根输出（基础）

单独跑一次第 2 节的小工作流，逐条打印每个事件的 `node_info.path`、`node_info.output_for` 和 `output`。

**验收**：用自己的话解释两件事——① 为什么**子节点也会产生"最终事件"**，所以不能只用 `is_final_response()` 取流程结果；② `output_for` 里"不含 `/`"的路径为什么指向根 Workflow。

### 3. 把 State 传值改成显式传值（进阶）

第 4 章的写作流水线用的是 `output_key` + instruction 里的 `{key}` 插值。把它改写成本章的 `ctx.run_node()` 返回值传值版本。

**验收**：列出改造后"哪些数据不再需要进 State"；并说明还有**哪一类数据仍然必须留在 State**。改造后流水线的模型调用次数不得增加。

### 4. 修订预算的三条路径（进阶）

扩展第 4 节的检查，务必覆盖三条路径：**一次通过**、**修改一次后通过**、**预算耗尽**。

**验收**：三条路径都断言 `stop_reason` 与 `revisions` 的具体取值；**预算耗尽必须返回失败**，不得把最后一稿标为成功。哪条路径调用了模型、调用了几次，也要断言出来。

### 5. 用数据证明并发上限（工程）

批量 20 项，`max_parallel_workers=3`，记录峰值并发、总耗时和返回顺序；再把上限改成 1 和 20 各跑一次。

**验收**：三次运行的峰值分别**不超过** 3 / 1 / 20；结果顺序**始终等于**输入顺序；用三次的数据说明"批次上限不等于服务配额"。（提示：上限为 1 时峰值应该是 1，这同时也是"串行"的对照组。）

参考实现见 [practice/](practice/) 目录下的 `c7_1.py` ~ `c7_5.py`。建议先自己写完再对照。

---

## ✅ 本次实际运行验收

In [9]:
import sys
from datetime import datetime, timedelta, timezone

expected = {
    "最小工作流与根输出",
    "真实模型流水线与执行顺序",
    "一次通过、预算耗尽与真实模型修改",
    "并发确实发生且不超过上限",
}
assert set(checks) == expected and all(checks.values())
assert len(model_responses) == 4
assert all(
    isinstance(r["tokens"], int) and r["tokens"] > 0 for r in model_responses
)
print(
    "执行时间：",
    datetime.now(timezone(timedelta(hours=8))).isoformat(timespec="seconds"),
)
print(
    "环境：Python",
    sys.version.split()[0],
    "/ google-adk",
    metadata.version("google-adk"),
)
for name in checks:
    print("✅", name)
print("真实 DeepSeek 响应次数：", len(model_responses))
print("服务返回的总 token 使用：", sum(r["tokens"] for r in model_responses))
print("✅ 第七章全部验收通过")

执行时间： 2026-10-14T00:31:17+08:00
环境：Python 3.12.3 / google-adk 2.6.3
✅ 最小工作流与根输出
✅ 真实模型流水线与执行顺序
✅ 一次通过、预算耗尽与真实模型修改
✅ 并发确实发生且不超过上限
真实 DeepSeek 响应次数： 4
服务返回的总 token 使用： 1305
✅ 第七章全部验收通过
